# AI Job Market Analytics

**Student:** Hemlata  
**Project:** AI Job Market Data Analysis  

This notebook analyzes the supplied AI Job Market Dataset. It covers data loading, cleaning, exploratory data analysis, salary and hiring trends, skill analysis, and a simple machine-learning model for salary prediction.

## 1. Import Libraries and Set Display Options

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 40)

print('Libraries imported successfully.')

## 2. Load the Dataset

In [ ]:
# Keep the CSV in the same folder as this notebook when running it locally.
DATA_FILE = 'AI Job Market Dataset.csv'

df = pd.read_csv(DATA_FILE)

print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
display(df.head())

## 3. Initial Data Inspection

In [ ]:
print('Column names:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes.to_frame('dtype'))

print('\nDataset information:')
df.info()

print('\nDuplicate rows:', df.duplicated().sum())
print('Missing values:', int(df.isna().sum().sum()))

## 4. Data Cleaning

In [ ]:
# Work on a copy so the original loaded dataset remains unchanged.
data = df.copy()

# Remove exact duplicate records.
data = data.drop_duplicates().reset_index(drop=True)

# Convert numeric columns safely and fill missing numeric values with the median.
numeric_cols = data.select_dtypes(include=np.number).columns
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')
    data[col] = data[col].fillna(data[col].median())

# Fill missing categorical values with the most frequent category.
categorical_cols = data.select_dtypes(include='object').columns
for col in categorical_cols:
    if data[col].isna().any():
        mode = data[col].mode(dropna=True)
        data[col] = data[col].fillna(mode.iloc[0] if not mode.empty else 'Unknown')

print(f'Rows after cleaning: {len(data):,}')
print(f'Remaining missing values: {int(data.isna().sum().sum())}')
print(f'Remaining duplicate rows: {data.duplicated().sum()}')

## 5. Descriptive Statistics

In [ ]:
display(data.describe(include='all').T)

## 6. Job Market Overview

The following summaries show the most common job titles, countries, industries, experience levels, remote-work types, and hiring urgency levels.

In [ ]:
summary_tables = {
    'Top Job Titles': data['job_title'].value_counts().head(10),
    'Top Countries': data['country'].value_counts().head(10),
    'Industries': data['company_industry'].value_counts(),
    'Experience Levels': data['experience_level'].value_counts(),
    'Remote Work Types': data['remote_type'].value_counts(),
    'Hiring Urgency': data['hiring_urgency'].value_counts()
}

for title, table in summary_tables.items():
    print(f'\n{title}')
    display(table.to_frame('count'))

## 7. Salary Analysis

In [ ]:
salary_stats = data['salary'].agg(['count', 'mean', 'median', 'min', 'max', 'std']).to_frame('salary')
display(salary_stats)

salary_by_experience = (
    data.groupby('experience_level', as_index=False)['salary']
        .agg(['count', 'mean', 'median', 'min', 'max'])
        .sort_values('mean', ascending=False)
)
print('Salary by experience level:')
display(salary_by_experience)

salary_by_remote = (
    data.groupby('remote_type', as_index=False)['salary']
        .agg(['count', 'mean', 'median'])
        .sort_values('mean', ascending=False)
)
print('Salary by remote-work type:')
display(salary_by_remote)

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(data['salary'], bins=30, edgecolor='black')
plt.title('Distribution of AI Job Salaries')
plt.xlabel('Salary')
plt.ylabel('Number of Job Postings')
plt.tight_layout()
plt.show()

In [ ]:
salary_plot = data.groupby('experience_level')['salary'].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(salary_plot.index, salary_plot.values)
plt.title('Average Salary by Experience Level')
plt.xlabel('Experience Level')
plt.ylabel('Average Salary')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 8. Job Demand and Hiring Analysis

In [ ]:
openings_by_title = (
    data.groupby('job_title')['job_openings']
        .sum()
        .sort_values(ascending=False)
        .head(10)
)

print('Top job titles by total openings:')
display(openings_by_title.to_frame('total_openings'))

plt.figure(figsize=(10, 5))
plt.bar(openings_by_title.index, openings_by_title.values)
plt.title('Top Job Titles by Total Job Openings')
plt.xlabel('Job Title')
plt.ylabel('Total Job Openings')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
yearly = data.groupby('job_posting_year').agg(
    postings=('job_id', 'count'),
    total_openings=('job_openings', 'sum'),
    average_salary=('salary', 'mean')
).sort_index()

display(yearly)

plt.figure(figsize=(10, 5))
plt.plot(yearly.index, yearly['postings'], marker='o')
plt.title('Job Postings by Year')
plt.xlabel('Posting Year')
plt.ylabel('Number of Postings')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Skill Demand Analysis

The skill columns are binary indicators: 1 means the skill is present/required in the job record and 0 means it is not.

In [ ]:
skill_cols = [
    'skills_python',
    'skills_sql',
    'skills_ml',
    'skills_deep_learning',
    'skills_cloud'
]

skill_counts = data[skill_cols].sum().sort_values(ascending=False)
skill_percent = (data[skill_cols].mean() * 100).sort_values(ascending=False).round(2)

skill_summary = pd.DataFrame({
    'job_postings_requiring_skill': skill_counts,
    'percentage_of_postings': skill_percent
})
display(skill_summary)

plt.figure(figsize=(9, 5))
plt.bar(skill_summary.index, skill_summary['percentage_of_postings'])
plt.title('Skill Demand in AI Job Postings')
plt.xlabel('Skill')
plt.ylabel('Percentage of Job Postings (%)')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 10. Education and Experience Analysis

In [ ]:
education_salary = data.groupby('education_level')['salary'].agg(['count', 'mean', 'median']).sort_values('mean', ascending=False)
display(education_salary)

experience_salary = data.groupby('years_experience')['salary'].mean().sort_index()

plt.figure(figsize=(10, 5))
plt.plot(experience_salary.index, experience_salary.values, marker='o')
plt.title('Average Salary vs. Years of Experience')
plt.xlabel('Years of Experience')
plt.ylabel('Average Salary')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Relationship Between Skills and Salary

In [ ]:
skill_salary = []
for skill in skill_cols:
    required = data.loc[data[skill] == 1, 'salary'].mean()
    not_required = data.loc[data[skill] == 0, 'salary'].mean()
    skill_salary.append({
        'skill': skill.replace('skills_', '').replace('_', ' ').title(),
        'avg_salary_when_required': required,
        'avg_salary_when_not_required': not_required,
        'difference': required - not_required
    })

skill_salary_df = pd.DataFrame(skill_salary).sort_values('difference', ascending=False)
display(skill_salary_df.round(2))

## 12. Correlation Analysis for Numeric Variables

In [ ]:
corr_cols = [
    'years_experience', 'skills_python', 'skills_sql', 'skills_ml',
    'skills_deep_learning', 'skills_cloud', 'salary', 'job_posting_year', 'job_openings'
]
correlation = data[corr_cols].corr().round(2)
display(correlation)

plt.figure(figsize=(10, 7))
plt.imshow(correlation, interpolation='nearest', aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=60, ha='right')
plt.yticks(range(len(correlation.index)), correlation.index)
plt.title('Correlation Matrix of Numeric Variables')
plt.tight_layout()
plt.show()

## 13. Machine Learning: Salary Prediction

A Random Forest regression model is trained to demonstrate a simple predictive analytics workflow. The model uses job characteristics available in the dataset and predicts salary.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_cols = [
    'job_title', 'company_size', 'company_industry', 'country', 'remote_type',
    'experience_level', 'years_experience', 'education_level',
    'skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning',
    'skills_cloud', 'job_posting_month', 'job_posting_year', 'hiring_urgency',
    'job_openings'
]

X = data[feature_cols]
y = data['salary']

categorical_features = X.select_dtypes(include='object').columns.tolist()
numeric_features = X.select_dtypes(exclude='object').columns.tolist()

preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('numeric', 'passthrough', numeric_features)
])

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'Mean Absolute Error (MAE): {mae:,.2f}')
print(f'Root Mean Squared Error (RMSE): {rmse:,.2f}')
print(f'R² Score: {r2:.3f}')

In [ ]:
comparison = pd.DataFrame({
    'Actual Salary': y_test.values,
    'Predicted Salary': predictions
})
display(comparison.head(10).round(2))

plt.figure(figsize=(7, 7))
plt.scatter(y_test, predictions, alpha=0.5)
min_val = min(y_test.min(), predictions.min())
max_val = max(y_test.max(), predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')
plt.title('Actual vs Predicted Salary')
plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.tight_layout()
plt.show()

## 14. Key Findings Generated from the Dataset

In [ ]:
top_job = data['job_title'].value_counts().idxmax()
top_country = data['country'].value_counts().idxmax()
top_skill = skill_counts.idxmax().replace('skills_', '').replace('_', ' ').title()
highest_avg_salary_level = data.groupby('experience_level')['salary'].mean().idxmax()
highest_avg_salary = data.groupby('experience_level')['salary'].mean().max()
top_openings_job = openings_by_title.idxmax()

print(f'1. Most frequently posted job title: {top_job}')
print(f'2. Country with the most job postings: {top_country}')
print(f'3. Most frequently required skill among the listed skills: {top_skill}')
print(f'4. Experience level with the highest average salary in this dataset: {highest_avg_salary_level} ({highest_avg_salary:,.2f})')
print(f'5. Job title with the highest total openings among the top-10 titles: {top_openings_job}')

print('\nNote: These findings describe this dataset and should not be interpreted as a universal description of the entire AI job market.')

## 15. Conclusion

The analysis demonstrates a complete data-analytics workflow: loading the AI job-market data, checking data quality, cleaning records, exploring job and salary patterns, studying skill demand, examining relationships among variables, and building a baseline salary-prediction model. The results can be used to understand patterns in the supplied dataset and to demonstrate practical Python data-analysis skills.

### Project Details
- **Student:** Hemlata
- **Dataset:** AI Job Market Dataset.csv
- **Main tools:** Python, Pandas, NumPy, Matplotlib, Scikit-learn
- **Project type:** Data Analytics with AI / Machine Learning demonstration